# Experiment Observatory

Deep analysis of AL experiment behavior beyond aggregate mAP.

## Sections
1. **Selection Analysis** — What images does each strategy pick?
2. **Selection Overlap** — How much do strategies agree?
3. **Class Distribution** — Are certain classes over/under-selected?
4. **Object Size Analysis** — Does AL prefer certain object sizes?
5. **Background Image Analysis** — How many empty images in each selection?

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

from observatory_utils import (
    VOC_CLASSES,
    STRATEGY_CHAINS_20_70,
    check_chain_monotonicity,
    class_dist_for_chain,
    class_distribution,
    diff_splits,
    full_pool_class_distribution,
    get_chain_additions,
    get_chain_splits,
    images_without_annotations,
    jaccard,
    object_size_stats,
    overlap_matrix_np,
    parse_split,
)

: 

## 1. Selection Analysis — What Gets Picked?

### 1b. Monotonicity Check — Are Selections Strictly Additive?

If a chain drops images between steps, it means the AL re-selects from scratch each iteration
rather than building on the previous selection. This is a data quality signal.

In [ ]:
for name, chain in STRATEGY_CHAINS_20_70.items():
    mono = check_chain_monotonicity(chain)
    if not mono:
        continue
    all_mono = all(v["monotonic"] for v in mono.values())
    status = "OK" if all_mono else "NON-MONOTONIC"
    print(f"--- {name} [{status}] ---")
    for step, info in mono.items():
        flag = "" if info["monotonic"] else " <<<< DROPPED IMAGES"
        print(
            f"  {step}: +{info['added']} added, -{info['dropped']} dropped, "
            f"{info['prev_size']}→{info['next_size']}{flag}"
        )
    print()

In [ ]:
# Show chain structure: how many images at each fraction, how many added per step
for name, chain in STRATEGY_CHAINS_20_70.items():
    splits = get_chain_splits(chain)
    if not splits:
        print(f"  {name}: no split files found")
        continue
    print(f"\n=== {name} ===")
    fracs = sorted(splits.keys())
    prev_set = None
    for f in fracs:
        cur_set = parse_split(splits[f])
        added = len(cur_set - prev_set) if prev_set is not None else len(cur_set)
        label = "(initial)" if prev_set is None else f"(+{added})"
        print(f"  {f:.1%}: {len(cur_set):>5} images {label}")
        prev_set = cur_set

## 2. Selection Overlap — Strategy Agreement

In [ ]:
# For each fraction step (e.g. 0.2→0.3), compare ADDED images across strategies
# All strategies share the same 0.2 base, so additions are what differ
fraction_pairs = [(0.2, 0.3), (0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 0.7)]

for prev_f, next_f in fraction_pairs:
    step_label = f"{prev_f}→{next_f}"
    additions_by_strategy = {}

    for name, chain in STRATEGY_CHAINS_20_70.items():
        splits = get_chain_splits(chain)
        if prev_f in splits and next_f in splits:
            added = diff_splits(splits[prev_f], splits[next_f])
            additions_by_strategy[name] = added

    if len(additions_by_strategy) < 2:
        continue

    mat, labels = overlap_matrix_np(additions_by_strategy)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        mat,
        xticklabels=labels,
        yticklabels=labels,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_title(f"Selection Overlap (Jaccard) — Step {step_label}")
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# Cumulative overlap: compare FULL selected sets at each fraction (not just additions)
for frac in [0.3, 0.4, 0.5, 0.6, 0.7]:
    sets_at_frac = {}
    for name, chain in STRATEGY_CHAINS_20_70.items():
        splits = get_chain_splits(chain)
        if frac in splits:
            sets_at_frac[name] = parse_split(splits[frac])

    if len(sets_at_frac) < 2:
        continue

    mat, labels = overlap_matrix_np(sets_at_frac)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        mat,
        xticklabels=labels,
        yticklabels=labels,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_title(f"Cumulative Set Overlap (Jaccard) at {frac:.0%}")
    plt.tight_layout()
    plt.show()

## 3. Class Distribution in Selected Samples

In [ ]:
# Full pool class distribution (reference)
pool_dist = full_pool_class_distribution()
total_annotations = sum(pool_dist.values())
pool_fracs = {cls: count / total_annotations for cls, count in pool_dist.items()}

print("Full pool class distribution:")
for cls_id in sorted(pool_dist.keys()):
    print(
        f"  {VOC_CLASSES[cls_id]:>15}: {pool_dist[cls_id]:>5} ({pool_fracs[cls_id]:.1%})"
    )
print(f"  {'TOTAL':>15}: {total_annotations}")

In [ ]:
# Per-step class distribution for each strategy
# Compare: what fraction of each class does each strategy add at each step?

strategies_to_compare = ["random", "distance", "distance_matryoshka_m", "density"]

for prev_f, next_f in [(0.2, 0.3), (0.3, 0.4), (0.4, 0.5)]:
    step_label = f"{prev_f}→{next_f}"
    fig, axes = plt.subplots(
        1, len(strategies_to_compare), figsize=(20, 6), sharey=True
    )
    fig.suptitle(f"Class Distribution of ADDED Images — Step {step_label}", fontsize=14)

    for ax, strategy_name in zip(axes, strategies_to_compare):
        chain = STRATEGY_CHAINS_20_70.get(strategy_name)
        if chain is None:
            continue
        splits = get_chain_splits(chain)
        if prev_f not in splits or next_f not in splits:
            ax.set_title(f"{strategy_name}\n(no data)")
            continue

        added = diff_splits(splits[prev_f], splits[next_f])
        dist = class_distribution(added)
        total = sum(dist.values())

        classes = sorted(VOC_CLASSES.keys())
        counts = [dist.get(c, 0) for c in classes]
        names = [VOC_CLASSES[c] for c in classes]
        # Normalize to fraction of additions
        fracs = [c / total if total > 0 else 0 for c in counts]
        pool_ref = [pool_fracs.get(c, 0) for c in classes]

        x = np.arange(len(classes))
        width = 0.35
        ax.barh(x - width / 2, fracs, width, label="Selected", color="steelblue")
        ax.barh(
            x + width / 2,
            pool_ref,
            width,
            label="Pool baseline",
            color="lightcoral",
            alpha=0.7,
        )
        ax.set_yticks(x)
        ax.set_yticklabels(names, fontsize=9)
        ax.set_xlabel("Fraction of annotations")
        ax.set_title(strategy_name)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# Class selection bias: ratio of (class fraction in selection) / (class fraction in pool)
# Values > 1 = overselected, < 1 = underselected

bias_data = []

for strategy_name in strategies_to_compare:
    chain = STRATEGY_CHAINS_20_70.get(strategy_name)
    if chain is None:
        continue
    splits = get_chain_splits(chain)
    fracs_available = sorted(splits.keys())

    for i in range(len(fracs_available) - 1):
        prev_f, next_f = fracs_available[i], fracs_available[i + 1]
        added = diff_splits(splits[prev_f], splits[next_f])
        dist = class_distribution(added)
        total = sum(dist.values())
        if total == 0:
            continue

        for cls_id in VOC_CLASSES:
            sel_frac = dist.get(cls_id, 0) / total
            pool_frac = pool_fracs.get(cls_id, 1e-9)
            bias_data.append(
                {
                    "strategy": strategy_name,
                    "step": f"{prev_f}→{next_f}",
                    "class": VOC_CLASSES[cls_id],
                    "class_id": cls_id,
                    "bias": sel_frac / pool_frac,
                }
            )

bias_df = pd.DataFrame(bias_data)

# Heatmap: average bias across all steps per strategy
avg_bias = bias_df.groupby(["strategy", "class"])["bias"].mean().unstack(level=0)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    avg_bias, annot=True, fmt=".2f", cmap="RdYlGn", center=1.0, ax=ax, linewidths=0.5
)
ax.set_title("Average Class Selection Bias (>1 = overselected, <1 = underselected)")
ax.set_ylabel("Class")
ax.set_xlabel("Strategy")
plt.tight_layout()
plt.show()

In [ ]:
# Key comparison: distance vs distance_matryoshka bias difference
# Positive = matryoshka selects MORE of this class than vanilla distance

if "distance" in avg_bias.columns and "distance_matryoshka_m" in avg_bias.columns:
    diff = avg_bias["distance_matryoshka_m"] - avg_bias["distance"]
    diff = diff.sort_values()

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ["#d32f2f" if v < 0 else "#388e3c" for v in diff.values]
    ax.barh(range(len(diff)), diff.values, color=colors)
    ax.set_yticks(range(len(diff)))
    ax.set_yticklabels(diff.index)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Bias difference (matryoshka − vanilla distance)")
    ax.set_title("Class Selection Shift: Matryoshka vs Vanilla Distance AL")
    plt.tight_layout()
    plt.show()
else:
    print("Missing data for distance vs matryoshka comparison")

## 4. Object Size Analysis

In [ ]:
# Compare object sizes in AL-selected vs random-selected additions
# Size = normalized w*h (proxy for object area)

for prev_f, next_f in [(0.2, 0.3), (0.3, 0.4)]:
    step_label = f"{prev_f}→{next_f}"
    size_data = []

    for strategy_name in strategies_to_compare:
        chain = STRATEGY_CHAINS_20_70.get(strategy_name)
        if chain is None:
            continue
        splits = get_chain_splits(chain)
        if prev_f not in splits or next_f not in splits:
            continue

        added = diff_splits(splits[prev_f], splits[next_f])
        sizes = object_size_stats(added)
        for cls_id, areas in sizes.items():
            for area in areas:
                size_data.append(
                    {
                        "strategy": strategy_name,
                        "area": area,
                        "class": VOC_CLASSES[cls_id],
                    }
                )

    if not size_data:
        continue

    size_df = pd.DataFrame(size_data)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Object Size Distribution — Step {step_label}")

    # Overall size distribution per strategy
    for strategy_name in strategies_to_compare:
        subset = size_df[size_df["strategy"] == strategy_name]
        if subset.empty:
            continue
        axes[0].hist(
            subset["area"], bins=50, alpha=0.5, label=strategy_name, density=True
        )
    axes[0].set_xlabel("Object area (normalized w×h)")
    axes[0].set_ylabel("Density")
    axes[0].set_title("All classes")
    axes[0].legend()
    axes[0].set_xlim(0, 0.3)

    # Box plot per strategy
    sns.boxplot(data=size_df, x="strategy", y="area", ax=axes[1], showfliers=False)
    axes[1].set_title("Object area by strategy")
    axes[1].set_ylabel("Area (normalized)")

    plt.tight_layout()
    plt.show()

## 5. Background Image Analysis

In [ ]:
# How many background (no-annotation) images does each strategy include?

bg_data = []
for strategy_name in STRATEGY_CHAINS_20_70:
    chain = STRATEGY_CHAINS_20_70[strategy_name]
    splits = get_chain_splits(chain)
    for frac, path in sorted(splits.items()):
        all_stems = parse_split(path)
        bg = images_without_annotations(all_stems)
        bg_data.append(
            {
                "strategy": strategy_name,
                "fraction": frac,
                "total_images": len(all_stems),
                "background_images": len(bg),
                "bg_ratio": len(bg) / len(all_stems) if all_stems else 0,
            }
        )

bg_df = pd.DataFrame(bg_data)
print(bg_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
for strategy_name in STRATEGY_CHAINS_20_70:
    subset = bg_df[bg_df["strategy"] == strategy_name]
    ax.plot(subset["fraction"], subset["bg_ratio"], marker="o", label=strategy_name)
ax.set_xlabel("Labeled fraction")
ax.set_ylabel("Background image ratio")
ax.set_title("Background Image Ratio Across AL Iterations")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Unique Selections — Strategy Signatures

Which images does one strategy select that others never would?

In [ ]:
# For the key comparison (distance vs distance_matryoshka_m),
# find images unique to each strategy's selections across ALL steps

key_strategies = ["distance", "distance_matryoshka_m", "random"]
all_additions = {}

for name in key_strategies:
    chain = STRATEGY_CHAINS_20_70.get(name)
    if chain is None:
        continue
    adds = get_chain_additions(chain)
    combined = set()
    for step_adds in adds.values():
        combined |= step_adds
    all_additions[name] = combined

if "distance" in all_additions and "distance_matryoshka_m" in all_additions:
    only_vanilla = all_additions["distance"] - all_additions["distance_matryoshka_m"]
    only_matryoshka = all_additions["distance_matryoshka_m"] - all_additions["distance"]
    shared = all_additions["distance"] & all_additions["distance_matryoshka_m"]

    print(f"Distance-only selections: {len(only_vanilla)}")
    print(f"Matryoshka-only selections: {len(only_matryoshka)}")
    print(f"Shared selections: {len(shared)}")
    print()

    # Class distribution of unique selections
    dist_vanilla = class_distribution(only_vanilla)
    dist_matr = class_distribution(only_matryoshka)

    classes = sorted(VOC_CLASSES.keys())
    class_names = [VOC_CLASSES[c] for c in classes]

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(classes))
    width = 0.35
    ax.bar(
        x - width / 2,
        [dist_vanilla.get(c, 0) for c in classes],
        width,
        label=f"Only in vanilla distance ({len(only_vanilla)} imgs)",
        color="steelblue",
    )
    ax.bar(
        x + width / 2,
        [dist_matr.get(c, 0) for c in classes],
        width,
        label=f"Only in matryoshka distance ({len(only_matryoshka)} imgs)",
        color="darkorange",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_ylabel("Annotation count")
    ax.set_title("Class Distribution of Strategy-Unique Selections")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Missing strategy data for comparison")

## 7. Summary Statistics Table

In [ ]:
# Per-strategy summary: annotations added, unique classes covered, avg object size

summary_rows = []
for strategy_name in strategies_to_compare:
    chain = STRATEGY_CHAINS_20_70.get(strategy_name)
    if chain is None:
        continue
    additions = get_chain_additions(chain)
    for step, stems in additions.items():
        dist = class_distribution(stems)
        sizes = object_size_stats(stems)
        all_areas = [a for areas in sizes.values() for a in areas]
        bg = images_without_annotations(stems)
        summary_rows.append(
            {
                "Strategy": strategy_name,
                "Step": step,
                "Images added": len(stems),
                "Annotations added": sum(dist.values()),
                "Classes present": len(dist),
                "Background images": len(bg),
                "Mean obj area": f"{np.mean(all_areas):.4f}" if all_areas else "N/A",
                "Median obj area": (
                    f"{np.median(all_areas):.4f}" if all_areas else "N/A"
                ),
            }
        )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)